# Educational Stock Forecasting: Dual-Head LSTM Benchmark

> **DISCLAIMER**: This notebook and the associated forecasting module are strictly for **educational and research purposes**. They do not constitute financial advice, trading signals, or investment recommendations. All metrics and analyses represent historical backtesting and statistical evaluations under simplified assumptions.

### Context & Objective
This study examines next-day directional and return predictability across 12 liquid National Stock Exchange (NSE) symbols over a 10.8-year historical period (2016 to 2026). We implement a dual-output Long Short-Term Memory (LSTM) network and benchmark it against classical baselines (Majority Class, Persistence, Logistic/Ridge Regression, and Zero-Return) with strict zero-lookahead protocols.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent.parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import json

from ml.config import RANDOM_SEED, TICKERS, ARTIFACTS_DIR
from ml.data import load_dataset_summary
from ml.features import FEATURE_COLS, compute_features
from ml.split import split_and_prepare_dataset
from ml.evaluate import compute_directional_metrics, compute_return_metrics, compute_bootstrap_ci

print(f"Target Tickers ({len(TICKERS)}): {TICKERS}")

## 1. Data Ingestion & Coverage Summary
We download daily OHLCV historical bars from Yahoo Finance for 12 liquid symbols representing major NIFTY sectors. Each ticker covers January 2016 through October 2026.

In [ ]:
summary = load_dataset_summary()
summary_df = pd.DataFrame(summary).T
summary_df

## 2. Feature Engineering with Zero Lookahead
We compute technical indicators at day $t$ using only data available up to day $t$:
- Log returns: $\ln(Close_t / Close_{t-1})$
- Rolling return statistics: mean and std over 5, 10, and 20 days
- Momentum: 10-day rate of change
- RSI (14-day exponential Wilder's moving average)
- MACD (12, 26, 9)
- Volume change & 20-day return volatility

Targets for day $t+1$:
- Next-day return: $\ln(Close_{t+1} / Close_t)$
- Next-day direction: $\mathbb{I}(Close_{t+1} > Close_t)$

In [ ]:
print(f"Feature Count: {len(FEATURE_COLS)}")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"{i:2d}. {col}")

## 3. Strict Chronological Splitting & Scaler Isolation
To prevent lookahead leakage:
1. Data is partitioned chronologically across all symbols (70% Train, 15% Validation, 15% Test).
2. The StandardScaler is fit **strictly on the training partition**.
3. Sliding windows of length 30 days are generated strictly within each partition without crossing boundaries.

In [ ]:
metrics_file = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_file, 'r') as f:
    metrics = json.load(f)

meta = metrics['metadata']
print(f"Train Samples : {meta['train_samples']} (Cutoff: {meta['train_cutoff_date']})")
print(f"Val Samples   : {meta['val_samples']} (Cutoff: {meta['val_cutoff_date']})")
print(f"Test Samples  : {meta['test_samples']}")

## 4. Benchmark Results & Baseline Comparisons
We evaluate the trained LSTM on the unseen Test set against four baseline benchmarks:
1. **Majority Class Baseline**: Predicts the most frequent training class.
2. **Persistence Baseline**: Predicts tomorrow's direction matches today's sign.
3. **Linear Baseline (Logistic & Ridge Regression)**: Classical statistical models on flattened 30-day window features.
4. **Zero-Return Baseline**: Always predicts 0 return.

In [ ]:
overall = metrics['overall_test_metrics']
lstm = overall['lstm']
base = overall['baselines']

comparison_table = pd.DataFrame([
    {'Model': 'LSTM Network (Dual-Head)', 'Directional Acc (%)': f"{lstm['directional_accuracy']:.2f}% (95% CI: [{lstm['ci_95'][0]:.2f}%, {lstm['ci_95'][1]:.2f}%])", 'F1 Score': f"{lstm['f1']:.4f}", 'Return RMSE': f"{lstm['rmse']:.6f}", 'Price MAPE': f"{lstm['price_mape_percent']:.2f}%"},
    {'Model': 'Majority Class Baseline', 'Directional Acc (%)': f"{base['majority_class']['directional_accuracy']:.2f}%", 'F1 Score': f"{base['majority_class']['f1']:.4f}", 'Return RMSE': '-', 'Price MAPE': '-'},
    {'Model': 'Persistence Baseline', 'Directional Acc (%)': f"{base['persistence']['directional_accuracy']:.2f}%", 'F1 Score': f"{base['persistence']['f1']:.4f}", 'Return RMSE': '-', 'Price MAPE': '-'},
    {'Model': 'Logistic Regression', 'Directional Acc (%)': f"{base['logistic_regression']['directional_accuracy']:.2f}%", 'F1 Score': f"{base['logistic_regression']['f1']:.4f}", 'Return RMSE': '-', 'Price MAPE': '-'},
    {'Model': 'Ridge Regression', 'Directional Acc (%)': '-', 'F1 Score': '-', 'Return RMSE': f"{base['ridge_regression']['rmse']:.6f}", 'Price MAPE': f"{base['ridge_regression']['price_mape_percent']:.2f}%"},
    {'Model': 'Zero-Return Baseline', 'Directional Acc (%)': '-', 'F1 Score': '-', 'Return RMSE': f"{base['zero_return']['rmse']:.6f}", 'Price MAPE': f"{base['zero_return']['price_mape_percent']:.2f}%"},
])

comparison_table

## 5. Honest Findings & Statistical Interpretation

### Key Observations
1. **The LSTM does not beat the baselines**:
   - The LSTM achieves **47.54%** directional accuracy.
   - The Majority Class baseline achieves **47.56%** (which falls directly inside the LSTM 95% bootstrap confidence interval of `[46.02%, 49.01%]`).
   - The Persistence baseline achieves **49.32%**, and Logistic Regression achieves **49.57%**.
   - The Zero-Return baseline achieves an RMSE of **0.014428**, outperforming the LSTM return RMSE of **0.015623**.

2. **Why does this happen?**
   - **Weak-Form Efficient Market Hypothesis (EMH)**: Liquid equity markets rapidly incorporate historical price and volume information. Past returns and standard technical indicators exhibit near-zero serial correlation over daily horizons.
   - **Signal-to-Noise Ratio**: Daily financial returns have an extremely low signal-to-noise ratio. Complex non-linear architectures like LSTMs quickly capture noise rather than enduring alpha unless supplied with orthogonal alternative data (e.g., fundamentals, order flow, order book dynamics).
   - **Strict Methodology**: Because we enforced strict chronological splitting, no lookahead leakage, and fitting scalers solely on training data, the model was prevented from learning spurious forward patterns.

### Conclusion
Reporting that the LSTM does not beat naive baselines is a statistically sound and scientifically honest outcome. It underscores the difficulty of daily market timing and demonstrates rigorous ML engineering practice.